In [2]:
from pyspark.sql import SparkSession, functions as F

BASE_PATH = "/work/data"   # no Airflow será /opt/airflow/data
YEAR_MONTH = "2026-01"

raw_path = f"{BASE_PATH}/raw/yellow_tripdata_{YEAR_MONTH}.parquet"
bronze_path = f"{BASE_PATH}/bronze/yellow_tripdata"

In [15]:
spark = SparkSession.builder.getOrCreate()

df_raw = spark.read.parquet(raw_path)

df_bronze = (
    df_raw
    .withColumn("_source_file", F.lit(f"yellow_tripdata_{YEAR_MONTH}.parquet"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_year_month", F.lit(YEAR_MONTH))
)

df_bronze.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (n

In [4]:
spark = SparkSession.builder.getOrCreate()

df_raw = spark.read.parquet(raw_path)

df_bronze = (
    df_raw
    .withColumn("_source_file", F.lit(f"yellow_tripdata_{YEAR_MONTH}.parquet"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_year_month", F.lit(YEAR_MONTH))
)

df_bronze.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (n

In [10]:
(
    df_bronze.write
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .partitionBy("_year_month")
    .parquet(bronze_path)
)

In [11]:
df_check = spark.read.parquet(bronze_path).filter(F.col("_year_month") == YEAR_MONTH)
n_bronze = df_check.count()
n_raw = df_raw.count()

print(n_raw, n_bronze, n_raw == n_bronze)

3724889 3724889 True


In [14]:
spark.stop()